### This demo showcases the implementation of stories 324 (Implement STAC view of PRIP products) and 761 (Implement staging of PRIP products) 

In [ ]:
# Init environment before running a demo notebook.
from resources.utils import *  
import pprint
init_demo()
# Reload the global vars again
from resources.utils import *  

from resources.dask_clusters.dask_main_env import *
await init_dask_cluster_staging()

pp = pprint.PrettyPrinter(indent=2, width=80, sort_dicts=False, compact=True)

#### Necessary roles: 
rs_prip_landing_page, rs_prip_s1a_read, rs_prip_s2b_read, RS_PROCESSES_STAGING_DOWNLOAD_S1A, RS_PROCESSES_STAGING_DOWNLOAD_S2B

In [ ]:
# STAC API landing page. /prip/
prip_client.get_landing()

In [ ]:
# STAC collections the user has permission to access. '/prip/collections'
collections = prip_client.get_collections()
for c in collections:
    pp.pprint(c.to_dict()['id'])

In [ ]:
# Queryable fields. '/prip/queryables'
general_queryables = prip_client.get_queryables()
assert isinstance(general_queryables, dict)
pprint.pp(general_queryables)

In [ ]:
fields = list(general_queryables["properties"].keys())
print(fields)

In [ ]:
# Request items from prip collection "S2B_L2A_TL"
items_collection_prip_s2b_l2a_tl = prip_client.search(max_items = 10, collections = ["S2B_L2A_TL"])
assert len(items_collection_prip_s2b_l2a_tl) != 0
# Request items from prip collection "S1A_L0_IW_RAW"
items_collection_prip_s1a_l0_iw = prip_client.search(max_items = 10, collections = ["S1A_L0_IW_RAW"])
assert len(items_collection_prip_s1a_l0_iw) != 0

In [ ]:
items_collection_prip_s2b_l2a_tl

In [ ]:
items_collection_prip_s1a_l0_iw

In [ ]:
filter = "product:type='IW_RAW__0N'"
prip_client.search(method='GET', stac_filter=filter)

In [ ]:
filter = "processing:facility='S1 Production Service-SERCO'"
items_collection_1_to_stage = prip_client.search(method='GET', stac_filter=filter)
items_collection_1_to_stage

In [ ]:
# sortby : id, prip:id, file:size, type, eviction_datetime, created, published, start_datetime, end_datetime

filter = "constellation='sentinel-2'"
sortby = "-id"
prip_client.search(method='GET', stac_filter=filter, sortby=sortby)

In [ ]:
filter="constellation='sentinel-2' AND intersects='POLYGON((103.0 80.0,-62 -10,-58 -10,-56 0,-60 0))'"
items_collection_2_to_stage = prip_client.search(method='GET', stac_filter=filter)
items_collection_2_to_stage

In [ ]:
stac_filter = {
  "op": "and",
  "args": [
    {
      "op": "=",
      "args": [
        { "property": "sat:orbit_state" },
        "ascending"
      ]
    }
  ]
}
prip_client.search(method='POST', stac_filter = stac_filter, collections = ["S1A_L0_IW_RAW"])

In [ ]:
# Create a test collection 
CATALOG_COLLECTION_ID = "SPRINT28_TEST_COLLECTION"
collection = create_test_collection(CATALOG_COLLECTION_ID)
items = catalog_client.get_items(CATALOG_COLLECTION_ID)
list(items)

### Starting 2 staging processes

In [ ]:
items_collection = [items_collection_1_to_stage, items_collection_2_to_stage]

staging_resp_list = []
for items in items_collection:
    staging_resp_list.append(staging_client.run_staging(items.to_dict(), CATALOG_COLLECTION_ID))

for resp in staging_resp_list:
    staging_client.wait_for_jobs(resp, logger)

In [ ]:
# Check that each of the job previously launched are successful
for entry in staging_resp_list:
    inner = next(iter(entry.values()))
    job_id = inner["jobID"]
    job_results = staging_client.get_job_results(job_id)
    print(f"Results from job {job_id}: {job_results}")
    assert job_results == "successful"

In [ ]:
result = list(catalog_client.get_collection(CATALOG_COLLECTION_ID).get_items())
result

### End of demo
Delete the whole collection

In [ ]:
result = catalog_client.remove_collection(CATALOG_COLLECTION_ID)
assert result.json()["deleted collection"] == CATALOG_COLLECTION_ID
pp.pprint(result.json())